In [1]:
# ============================================================
# DAY 13 — PORTFOLIO MANAGER
# AI Stock Screener — Indian Markets
# ============================================================

import pandas as pd
import numpy as np
import os
from datetime import datetime
from difflib import get_close_matches

# ── PATHS ─────────────────────────────────────────────────────
BASE_DIR      = r"E:\learn\Project 1 AI Screener\stock-ai-india"
DATA_DIR      = os.path.join(BASE_DIR, 'data')
UNIVERSE_DIR  = os.path.join(DATA_DIR, 'universe')
FUND_DIR      = os.path.join(DATA_DIR, 'fundamentals')
SCORES_DIR    = os.path.join(DATA_DIR, 'scores')
PORTFOLIO_DIR = os.path.join(DATA_DIR, 'portfolio')
REPORTS_DIR   = os.path.join(DATA_DIR, 'reports', 'portfolio')

for d in [PORTFOLIO_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── LOAD DATA ─────────────────────────────────────────────────
tech_df    = pd.read_csv(os.path.join(SCORES_DIR,   'technical_report_full.csv'))
fund_df    = pd.read_csv(os.path.join(FUND_DIR,     'fundamental_scores_full.csv'))
prefilt_df = pd.read_csv(os.path.join(UNIVERSE_DIR, 'prefilt_passed.csv'))
quality_df = pd.read_csv(os.path.join(UNIVERSE_DIR, 'quality_passed.csv'))

# Merge Market Cap into fund_df
if 'Market_Cap_Cr' not in fund_df.columns:
    fund_df = fund_df.merge(
        prefilt_df[['Symbol', 'Market_Cap_Cr']], on='Symbol', how='left'
    )

# Valid symbol set
valid_symbols = set(quality_df['Symbol'].tolist())

print(f"✅ Data loaded")
print(f"   Tech report  : {len(tech_df)} stocks")
print(f"   Fund scores  : {len(fund_df)} stocks")
print(f"   Valid symbols: {len(valid_symbols)}")
print(f"\n   Tech columns : {list(tech_df.columns)}")

✅ Data loaded
   Tech report  : 752 stocks
   Fund scores  : 752 stocks
   Valid symbols: 752

   Tech columns : ['Symbol', 'Sector', 'Fund Score', 'Market Cap Cr', 'Cap Category', 'Current Price', 'RSI', 'ADX', 'MACD Hist', 'Vol Ratio', 'EMA20', 'EMA50', 'EMA200', 'DI_Plus', 'DI_Minus', 'Momentum Score', 'Reversal Score', 'Best Setup', 'Tech Score', 'Tier', 'In Consolidation', 'Consol Days', 'Consol Label', 'Pct to Breakout', 'Consol Volume', 'ML_Prediction', 'ML_Confidence', 'Forecast_25d_Pct', 'Forecast_45d_Pct', 'Forecast_180d_Pct', 'Forecast_25d_Price', 'Forecast_45d_Price', 'Forecast_180d_Price', 'Bottom_Rev_Prob', 'Top_Rev_Prob', 'Bottom_Rev_Flag', 'Top_Rev_Flag', 'Bullish_Cont_Prob', 'Bearish_Cont_Prob', 'Sector Score', 'Cap Score', 'Vol 5D Ratio', 'Breakout Vol', 'Vol Label', 'Vol Inference', 'Sector Trend', 'Sector Detail', 'Weeks_Count', 'First_Breakout_Date']


In [3]:
# ── CELL 2 : HELPER FUNCTIONS + SCORING LOGIC ─────────────────

def mcap_str(mcap_cr):
    if mcap_cr is None or pd.isna(mcap_cr):
        return '—'
    if mcap_cr >= 100000:
        return f'Rs{mcap_cr/100000:.1f}L Cr'
    elif mcap_cr >= 1000:
        return f'Rs{mcap_cr:,.0f}Cr'
    else:
        return f'Rs{mcap_cr:.0f}Cr'

def tier_abbr(tier):
    tier = str(tier)
    if 'BUY NOW (Momentum)' in tier: return 'T1-Mom'
    if 'BUY NOW (Reversal)' in tier: return 'T1-Rev'
    if 'BREAKOUT IMMINENT'  in tier: return 'T1-Brk'
    if 'WATCHLIST'          in tier: return 'T2-Wtch'
    if 'BASE BUILDING'      in tier: return 'T2-Base'
    return 'T3'

def is_sector_bullish(row):
    trend = str(row.get('Sector Trend', ''))
    return 'Uptrend' in trend

def is_sector_strong_bull(row):
    trend = str(row.get('Sector Trend', ''))
    return 'Strong Uptrend' in trend

def passes_ema_filter(row):
    """
    Hard filter for Tier 1 and Tier 2:
    Price > EMA50 > EMA200
    Ensures genuine uptrend — not just a bounce from lows.
    """
    try:
        price  = float(row.get('Current Price', 0) or 0)
        ema50  = float(row.get('EMA50',  0) or 0)
        ema200 = float(row.get('EMA200', 0) or 0)
        return price > ema50 > ema200
    except:
        return False

def get_priority_tier(row, for_swing=False):
    """
    Priority tiers for recommendation ranking.

    Tier 1 (Best): Sector bullish + ML Bullish Continual
                   + Technical Momentum + Price > EMA50 > EMA200
    Tier 2       : ML Bullish Continual + Technical Momentum
                   + Price > EMA50 > EMA200 (sector not bullish)
    Tier 3 (LT only): ML Reversal + Technical Reversal
                   + Fund Score >= 65 (no EMA filter — contrarian)

    Returns 1, 2, 3 or None if does not qualify.
    """
    ml_pred    = str(row.get('ML_Prediction', ''))
    best_setup = str(row.get('Best Setup', ''))
    fund_score = float(row.get('Fund Score', 0) or 0)
    ema_ok     = passes_ema_filter(row)

    # Tier 1: All three aligned + EMA confirmed
    if (ml_pred    == 'Bullish Continual' and
        best_setup == 'Momentum' and
        ema_ok and
        is_sector_bullish(row)):
        return 1

    # Tier 2: ML + Tech aligned + EMA confirmed, sector not bullish
    if (ml_pred    == 'Bullish Continual' and
        best_setup == 'Momentum' and
        ema_ok):
        return 2

    # Tier 3: Reversal — LT only, high fund score, no EMA filter
    if not for_swing:
        if (ml_pred    == 'Reversal' and
            best_setup == 'Reversal' and
            fund_score >= 65):
            return 3

    return None  # does not qualify

def compute_priority_score(row):
    """
    Sort score within each priority tier.
    Base  : 50% Sector Rank + 50% Cap Rank
    Bonus : +2 if sector strong uptrend, +1 if sector uptrend
    Boost : small ML confidence boost (max 0.5 pts)
    """
    sec_rank = float(row.get('Sector Score',   0) or 0)
    cap_rank = float(row.get('Cap Score',      0) or 0)
    ml_conf  = float(row.get('ML_Confidence', 0) or 0)

    base = sec_rank * 0.5 + cap_rank * 0.5

    if is_sector_strong_bull(row):
        bonus = 2.0
    elif is_sector_bullish(row):
        bonus = 1.0
    else:
        bonus = 0.0

    conf_boost = min(ml_conf / 200, 0.5)

    return round(base + bonus + conf_boost, 3)

# ── VERIFY ────────────────────────────────────────────────────
print("EMA filter check on AAVAS and AARTIPHARM:")
for sym in ['AAVAS', 'AARTIPHARM']:
    row = tech_df[tech_df['Symbol'] == sym]
    if len(row) > 0:
        r    = row.iloc[0]
        ema_ok = passes_ema_filter(r)
        pt   = get_priority_tier(r)
        print(f"  {sym:<14} Price={r['Current Price']:.2f}  "
              f"EMA50={r['EMA50']:.2f}  EMA200={r['EMA200']:.2f}  "
              f"EMA filter={ema_ok}  → Priority Tier={pt}")

print()
# Recount with EMA filter
t1 = sum(1 for _, r in tech_df.iterrows() if get_priority_tier(r) == 1)
t2 = sum(1 for _, r in tech_df.iterrows() if get_priority_tier(r) == 2)
t3 = sum(1 for _, r in tech_df.iterrows() if get_priority_tier(r) == 3)
print(f"Universe breakdown (with EMA filter):")
print(f"  Tier 1 (Sector+ML+Tech+EMA all confirmed) : {t1} stocks")
print(f"  Tier 2 (ML+Tech+EMA confirmed, sector not) : {t2} stocks")
print(f"  Tier 3 (Reversal, LT only, no EMA filter)  : {t3} stocks")

EMA filter check on AAVAS and AARTIPHARM:
  AAVAS          Price=1243.00  EMA50=1241.72  EMA200=1480.52  EMA filter=False  → Priority Tier=None
  AARTIPHARM     Price=690.35  EMA50=686.25  EMA200=743.64  EMA filter=False  → Priority Tier=None

Universe breakdown (with EMA filter):
  Tier 1 (Sector+ML+Tech+EMA all confirmed) : 34 stocks
  Tier 2 (ML+Tech+EMA confirmed, sector not) : 106 stocks
  Tier 3 (Reversal, LT only, no EMA filter)  : 4 stocks


In [4]:
# ── CELL 2B : VISUAL INSPECTION — TOP 3 PER TIER ─────────────

CAP_ORDER = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']

# Build scored dataframe
tech_df['Priority_Tier']  = tech_df.apply(lambda r: get_priority_tier(r), axis=1)
tech_df['Priority_Score'] = tech_df.apply(compute_priority_score, axis=1)

for tier_num in [1, 2, 3]:
    label = {
        1: 'TIER 1 — Sector + ML + Tech + EMA all confirmed',
        2: 'TIER 2 — ML + Tech + EMA confirmed (sector not bullish)',
        3: 'TIER 3 — Reversal (LT only, high fund score)',
    }[tier_num]

    subset = tech_df[tech_df['Priority_Tier'] == tier_num].sort_values(
        'Priority_Score', ascending=False
    ).head(3)

    print(f"\n{'='*90}")
    print(f"  {label}  ({len(tech_df[tech_df['Priority_Tier']==tier_num])} total)")
    print(f"{'='*90}")
    print(f"  {'Symbol':<14} {'Sector':<30} {'Cap':<16} "
          f"{'SecRnk':>6}  {'CapRnk':>6}  "
          f"{'Price':>8}  {'EMA50':>8}  {'EMA200':>8}  "
          f"{'ML Pred':<20} {'Conf':>5}  "
          f"{'Setup':<10} {'Sector Trend':<22} {'Score':>6}")
    print(f"  {'─'*14} {'─'*30} {'─'*16} "
          f"{'─'*6}  {'─'*6}  "
          f"{'─'*8}  {'─'*8}  {'─'*8}  "
          f"{'─'*20} {'─'*5}  "
          f"{'─'*10} {'─'*22} {'─'*6}")

    for _, row in subset.iterrows():
        sec_bull = '✅' if is_sector_bullish(row) else '  '
        print(f"  {row['Symbol']:<14} "
              f"{str(row.get('Sector','?')):<30} "
              f"{str(row.get('Cap Category','?')):<16} "
              f"{row.get('Sector Score',0):>6.1f}  "
              f"{row.get('Cap Score',0):>6.1f}  "
              f"{row.get('Current Price',0):>8.2f}  "
              f"{row.get('EMA50',0):>8.2f}  "
              f"{row.get('EMA200',0):>8.2f}  "
              f"{str(row.get('ML_Prediction','')):<20} "
              f"{float(row.get('ML_Confidence',0) or 0):>5.1f}  "
              f"{str(row.get('Best Setup','')):<10} "
              f"{sec_bull}{str(row.get('Sector Trend','')):<20} "
              f"{row.get('Priority_Score',0):>6.3f}")


  TIER 1 — Sector + ML + Tech + EMA all confirmed  (34 total)
  Symbol         Sector                         Cap              SecRnk  CapRnk     Price     EMA50    EMA200  ML Pred               Conf  Setup      Sector Trend            Score
  ────────────── ────────────────────────────── ──────────────── ──────  ──────  ────────  ────────  ────────  ──────────────────── ─────  ────────── ────────────────────── ──────
  LLOYDSME       Metals & Mining                Large Cap          10.0    10.0   1510.00   1275.94   1265.55  Bullish Continual     76.3  Momentum   ✅Strong Uptrend ↑↑    12.382
  NATIONALUM     Metals & Mining                Large Cap           9.2     9.2    418.00    371.63    291.81  Bullish Continual     81.1  Momentum   ✅Strong Uptrend ↑↑    11.605
  SARDAEN        Metals & Mining                Mini Large Cap      8.2    10.0    556.00    520.54    506.15  Bullish Continual     76.7  Momentum   ✅Strong Uptrend ↑↑    11.483

  TIER 2 — ML + Tech + EMA confirmed (s

In [10]:
# ── CELL 3 : LONG TERM RECOMMENDATIONS (ML Confidence >= 40) ──

TOP_N        = 10
MAX_REVERSAL = 2
MIN_CONF     = 40.0
CAP_ORDER    = ['Large Cap', 'Mini Large Cap', 'Mid Cap', 'Small Cap']

SECTOR_SHORT = {
    'Information Technology'            : 'IT',
    'Financial Services'                : 'Financial',
    'Chemicals'                         : 'Chemicals',
    'Healthcare'                        : 'Healthcare',
    'Consumer Durables'                 : 'Con Durables',
    'Consumer Services'                 : 'Con Services',
    'Fast Moving Consumer Goods'        : 'FMCG',
    'Capital Goods'                     : 'Capital Goods',
    'Automobile and Auto Components'    : 'Auto',
    'Construction'                      : 'Construction',
    'Construction Materials'            : 'Const Matrl',
    'Textiles'                          : 'Textiles',
    'Services'                          : 'Services',
    'Metals & Mining'                   : 'Metals',
    'Oil, Gas & Consumable Fuels'       : 'Oil & Gas',
    'Power'                             : 'Power',
    'Realty'                            : 'Realty',
    'Utilities'                         : 'Utilities',
    'Telecommunication'                 : 'Telecom',
    'Media, Entertainment & Publication': 'Media',
    'Media Entertainment & Publication' : 'Media',
    'Forest Materials'                  : 'Forest Matrl',
    'Diversified'                       : 'Diversified',
    'Pharmaceuticals'                   : 'Pharma',
    'Banking'                           : 'Banking',
    'Cement'                            : 'Cement',
    'Defence'                           : 'Defence',
    'Agriculture'                       : 'Agriculture',
}

SECTOR_TREND_SHORT = {
    'Strong Uptrend ↑↑'  : 'Str Up ↑↑',
    'Uptrend ↑'          : 'Uptrend ↑',
    'Weak Uptrend →↑'    : 'Wk Up →↑',
    'Sideways →'         : 'Sideways →',
    'Weak Downtrend →↓'  : 'Wk Dn →↓',
    'Downtrend ↓'        : 'Downtrend ↓',
    'Strong Downtrend ↓↓': 'Str Dn ↓↓',
    'No data'            : 'No data',
}

def short_sector(name):
    return SECTOR_SHORT.get(str(name), str(name)[:14])

def short_trend(trend):
    t = str(trend)
    for k, v in SECTOR_TREND_SHORT.items():
        if k in t:
            return v
    return t[:12]

# ── APPLY CONFIDENCE FILTER ───────────────────────────────────
# For Tier 1 + 2: ML_Confidence >= 40 (model actually validated)
# For Tier 3 reversal: Bottom_Rev_Prob >= 40
tech_df['Effective_Conf'] = tech_df.apply(
    lambda r: float(r.get('Bottom_Rev_Prob', 0) or 0)
              if str(r.get('ML_Prediction','')) == 'Reversal'
              else float(r.get('ML_Confidence', 0) or 0),
    axis=1
)

lt_lines = []
def p(line=''): lt_lines.append(str(line))

p("=" * 108)
p("  LONG TERM PORTFOLIO — STOCK RECOMMENDATIONS")
p(f"  Generated : {datetime.now().strftime('%Y-%m-%d %H:%M')}  |  "
  f"Universe: {len(tech_df)} stocks  |  ML Confidence filter: >={MIN_CONF}%")
p()
p("  Priority:")
p("  ★★★ Tier 1 : Sector bullish + ML Bullish Continual + Tech Momentum + Price > EMA50 > EMA200")
p("  ★★☆ Tier 2 : ML Bullish Continual + Tech Momentum + Price > EMA50 > EMA200 (sector not bullish)")
p("  ★☆☆ Tier 3 : ML Reversal + Tech Reversal + Fund Score >= 65 (contrarian, max 2 per cap)")
p(f"  Note: Only stocks with ML Confidence >= {MIN_CONF}% shown (model-validated signals only)")
p("=" * 108)

for cap in CAP_ORDER:
    cap_lbl = {'Large Cap':'L','Mini Large Cap':'ML',
               'Mid Cap':'M','Small Cap':'S'}[cap]

    # Tier 1 with confidence filter
    t1_cap = tech_df[
        (tech_df['Priority_Tier']  == 1) &
        (tech_df['Cap Category']   == cap) &
        (tech_df['Effective_Conf'] >= MIN_CONF)
    ].sort_values('Priority_Score', ascending=False)

    # Tier 2 with confidence filter
    t2_cap = tech_df[
        (tech_df['Priority_Tier']  == 2) &
        (tech_df['Cap Category']   == cap) &
        (tech_df['Effective_Conf'] >= MIN_CONF)
    ].sort_values('Priority_Score', ascending=False)

    # Tier 3 with confidence filter
    t3_cap = tech_df[
        (tech_df['Priority_Tier']  == 3) &
        (tech_df['Cap Category']   == cap) &
        (tech_df['Effective_Conf'] >= MIN_CONF)
    ].sort_values('Priority_Score', ascending=False).head(MAX_REVERSAL)

    bc_slots  = TOP_N - len(t3_cap)
    combined  = pd.concat([t1_cap, t2_cap]).head(bc_slots)
    all_picks = pd.concat([combined, t3_cap])

    if len(all_picks) == 0:
        p(f"\n  [{cap_lbl}] {cap}  —  No qualifying stocks with ML confidence >= {MIN_CONF}%")
        continue

    p(f"\n{'─'*108}")
    p(f"  [{cap_lbl}] {cap}  —  "
      f"{len(t1_cap)} Tier1  |  {len(t2_cap)} Tier2  |  "
      f"{len(t3_cap)} Tier3 reversal  "
      f"(showing top {min(len(all_picks), TOP_N)})")
    p(f"{'─'*108}")
    p(f"  {'#':<3}  {'★':<3}  {'Symbol':<12} {'Sector':<14} "
      f"{'Sec Trend':<12} {'ML Prediction':<20} {'Conf':>5}  "
      f"{'Setup':<9} {'Tech':>4}  {'SecRnk':>6}  {'CapRnk':>6}  "
      f"{'Price':>8}  {'MCap':>12}")
    p(f"  {'─'*3}  {'─'*3}  {'─'*12} {'─'*14} "
      f"{'─'*12} {'─'*20} {'─'*5}  "
      f"{'─'*9} {'─'*4}  {'─'*6}  {'─'*6}  "
      f"{'─'*8}  {'─'*12}")

    serial = 1
    for _, row in all_picks.iterrows():
        tier_n   = int(row.get('Priority_Tier', 0) or 0)
        stars    = {1:'★★★', 2:'★★☆', 3:'★☆☆'}.get(tier_n, '   ')
        sec_bull = '✅' if is_sector_bullish(row) else '  '
        conf     = float(row.get('Effective_Conf', 0) or 0)
        price    = float(row.get('Current Price', 0) or 0)
        mcap     = float(row.get('Market Cap Cr', 0) or 0)

        p(f"  {serial:<3}  {stars}  {row['Symbol']:<12} "
          f"{short_sector(row.get('Sector','?')):<14} "
          f"{sec_bull}{short_trend(row.get('Sector Trend','')):<11} "
          f"{str(row.get('ML_Prediction','')):<20} "
          f"{conf:>5.1f}  "
          f"{str(row.get('Best Setup','')):<9} "
          f"{float(row.get('Tech Score',0) or 0):>4.0f}  "
          f"{float(row.get('Sector Score',0) or 0):>6.1f}  "
          f"{float(row.get('Cap Score',0) or 0):>6.1f}  "
          f"{price:>8.2f}  "
          f"{mcap_str(mcap):>12}")
        serial += 1

p(f"\n{'─'*108}")
p(f"  ★★★ Sector+ML+Tech all bullish  |  "
  f"★★☆ ML+Tech bullish, sector not  |  ★☆☆ Reversal")
p(f"  ✅ Sector in uptrend  |  "
  f"Conf = ML Confidence (Bottom Rev Prob for reversals)  |  "
  f"Only model-validated signals shown")
p(f"{'─'*108}")

for line in lt_lines:
    print(line)

  LONG TERM PORTFOLIO — STOCK RECOMMENDATIONS
  Generated : 2026-04-15 10:11  |  Universe: 752 stocks  |  ML Confidence filter: >=40.0%

  Priority:
  ★★★ Tier 1 : Sector bullish + ML Bullish Continual + Tech Momentum + Price > EMA50 > EMA200
  ★★☆ Tier 2 : ML Bullish Continual + Tech Momentum + Price > EMA50 > EMA200 (sector not bullish)
  ★☆☆ Tier 3 : ML Reversal + Tech Reversal + Fund Score >= 65 (contrarian, max 2 per cap)
  Note: Only stocks with ML Confidence >= 40.0% shown (model-validated signals only)

────────────────────────────────────────────────────────────────────────────────────────────────────────────
  [L] Large Cap  —  15 Tier1  |  20 Tier2  |  2 Tier3 reversal  (showing top 10)
────────────────────────────────────────────────────────────────────────────────────────────────────────────
  #    ★    Symbol       Sector         Sec Trend    ML Prediction         Conf  Setup     Tech  SecRnk  CapRnk     Price          MCap
  ───  ───  ──────────── ────────────── ────────

In [13]:
# ── CELL 4 : SWING TRADING RECOMMENDATIONS ────────────────────
# Bullish Continual only | No reversals | Fund Score >= 40
# ML Confidence >= 40 | Price > EMA50 > EMA200
# Top 10 per cap category

MIN_FUND_SWING = 40.0
SWING_TOP_N    = 10

def get_priority_tier_swing(row):
    """
    Swing priority — no reversals, lower fund score threshold.
    Tier 1: Sector bullish + ML Bullish Continual + Tech Momentum
            + Price > EMA50 > EMA200 + Fund Score >= 40
    Tier 2: ML Bullish Continual + Tech Momentum
            + Price > EMA50 > EMA200 + Fund Score >= 40
    """
    ml_pred    = str(row.get('ML_Prediction', ''))
    best_setup = str(row.get('Best Setup', ''))
    fund_score = float(row.get('Fund Score', 0) or 0)
    ema_ok     = passes_ema_filter(row)
    conf_ok    = float(row.get('ML_Confidence', 0) or 0) >= MIN_CONF

    if fund_score < MIN_FUND_SWING:
        return None
    if not ema_ok:
        return None
    if not conf_ok:
        return None
    if ml_pred != 'Bullish Continual':
        return None
    if best_setup != 'Momentum':
        return None

    if is_sector_bullish(row):
        return 1
    return 2

# Apply swing priority
tech_df['Swing_Priority_Tier']  = tech_df.apply(
    lambda r: get_priority_tier_swing(r), axis=1
)
tech_df['Swing_Priority_Score'] = tech_df.apply(
    compute_priority_score, axis=1
)

# Count
sw_t1 = tech_df[tech_df['Swing_Priority_Tier'] == 1]
sw_t2 = tech_df[tech_df['Swing_Priority_Tier'] == 2]
print(f"Swing universe:")
print(f"  Tier 1 (Sector+ML+Tech bullish) : {len(sw_t1)} stocks")
print(f"  Tier 2 (ML+Tech bullish)         : {len(sw_t2)} stocks")
print(f"\nBreakdown by cap:")
for cap in CAP_ORDER:
    t1 = len(sw_t1[sw_t1['Cap Category'] == cap])
    t2 = len(sw_t2[sw_t2['Cap Category'] == cap])
    print(f"  {cap:<16} : T1={t1}  T2={t2}  Total={t1+t2}")

Swing universe:
  Tier 1 (Sector+ML+Tech bullish) : 20 stocks
  Tier 2 (ML+Tech bullish)         : 59 stocks

Breakdown by cap:
  Large Cap        : T1=15  T2=18  Total=33
  Mini Large Cap   : T1=4  T2=23  Total=27
  Mid Cap          : T1=1  T2=14  Total=15
  Small Cap        : T1=0  T2=4  Total=4


In [17]:
# ── CELL 5 : SWING TRADING RECOMMENDATIONS — DISPLAY ──────────

swing_lines = []
def ps(line=''): swing_lines.append(str(line))

ps("=" * 106)
ps("  SWING TRADING PORTFOLIO — STOCK RECOMMENDATIONS")
ps(f"  Generated : {datetime.now().strftime('%Y-%m-%d %H:%M')}  |  "
   f"Universe: {len(tech_df)} stocks  |  ML Confidence filter: >={MIN_CONF}%")
ps()
ps("  Priority:")
ps("  ★★★ Tier 1 : Sector bullish + ML Bull Cont + Tech Momentum + Price > EMA50 > EMA200")
ps("  ★★☆ Tier 2 : ML Bull Cont + Tech Momentum + Price > EMA50 > EMA200 (sector not bullish)")
ps(f"  Note: Bull Cont only | No reversals | Fund Score >= {MIN_FUND_SWING} | 30-day cycle")
ps(f"  [LT] = also in Long Term recommendations")
ps("=" * 106)

for cap in CAP_ORDER:
    cap_lbl = {'Large Cap':'L','Mini Large Cap':'ML',
               'Mid Cap':'M','Small Cap':'S'}[cap]

    t1_cap = tech_df[
        (tech_df['Swing_Priority_Tier']  == 1) &
        (tech_df['Cap Category']         == cap)
    ].sort_values('Swing_Priority_Score', ascending=False)

    t2_cap = tech_df[
        (tech_df['Swing_Priority_Tier']  == 2) &
        (tech_df['Cap Category']         == cap)
    ].sort_values('Swing_Priority_Score', ascending=False)

    all_picks = pd.concat([t1_cap, t2_cap]).head(SWING_TOP_N)

    if len(all_picks) == 0:
        ps(f"\n  [{cap_lbl}] {cap}  —  No qualifying stocks")
        continue

    lt_overlap = sum(1 for s in all_picks['Symbol'] if s in lt_symbols)

    ps(f"\n{'─'*106}")
    ps(f"  [{cap_lbl}] {cap}  —  "
       f"{len(t1_cap)} Tier1  |  {len(t2_cap)} Tier2  "
       f"(showing top {min(len(all_picks), SWING_TOP_N)})  "
       f"[LT overlap: {lt_overlap}]")
    ps(f"{'─'*106}")
    ps(f"  {'#':<3}  {'★':<3}  {'Symbol':<12} {'LT':<4} {'Sector':<14} "
       f"{'Sec Trend':<12} {'ML':<10} {'Conf':>5}  "
       f"{'Tech':>4}  {'SecRnk':>6}  {'CapRnk':>6}  "
       f"{'Price':>8}  {'MCap':>12}  {'25d%':>5}  {'45d%':>5}")
    ps(f"  {'─'*3}  {'─'*3}  {'─'*12} {'─'*4} {'─'*14} "
       f"{'─'*12} {'─'*10} {'─'*5}  "
       f"{'─'*4}  {'─'*6}  {'─'*6}  "
       f"{'─'*8}  {'─'*12}  {'─'*5}  {'─'*5}")

    serial = 1
    for _, row in all_picks.iterrows():
        tier_n   = int(row.get('Swing_Priority_Tier', 0) or 0)
        stars    = {1:'★★★', 2:'★★☆'}.get(tier_n, '   ')
        sec_bull = '✅' if is_sector_bullish(row) else '  '
        conf     = float(row.get('ML_Confidence', 0) or 0)
        price    = float(row.get('Current Price', 0) or 0)
        mcap     = float(row.get('Market Cap Cr', 0) or 0)
        f25      = float(row.get('Forecast_25d_Pct', 0) or 0)
        f45      = float(row.get('Forecast_45d_Pct', 0) or 0)
        lt_tag   = '[LT]' if row['Symbol'] in lt_symbols else '    '

        ps(f"  {serial:<3}  {stars}  {row['Symbol']:<12} {lt_tag:<4} "
           f"{short_sector(row.get('Sector','?')):<14} "
           f"{sec_bull}{short_trend(row.get('Sector Trend','')):<11} "
           f"{short_ml(row.get('ML_Prediction','')):<10} "
           f"{conf:>5.1f}  "
           f"{float(row.get('Tech Score',0) or 0):>4.0f}  "
           f"{float(row.get('Sector Score',0) or 0):>6.1f}  "
           f"{float(row.get('Cap Score',0) or 0):>6.1f}  "
           f"{price:>8.2f}  "
           f"{mcap_str(mcap):>12}  "
           f"{f25:>+5.1f}  "
           f"{f45:>+5.1f}")
        serial += 1

ps(f"\n{'─'*106}")
ps(f"  ★★★ Sector+ML+Tech all bullish  |  "
   f"★★☆ ML+Tech bullish, sector not  |  ✅ Sector uptrend")
ps(f"  Bull Cont = Bullish Continual  |  "
   f"25d%/45d% = ML forecast  |  No reversals in swing")
ps(f"  [LT] = stock also in Long Term recommendations — "
   f"consider for diversification")
ps(f"{'─'*106}")

for line in swing_lines:
    print(line)

  SWING TRADING PORTFOLIO — STOCK RECOMMENDATIONS
  Generated : 2026-04-15 10:36  |  Universe: 752 stocks  |  ML Confidence filter: >=40.0%

  Priority:
  ★★★ Tier 1 : Sector bullish + ML Bull Cont + Tech Momentum + Price > EMA50 > EMA200
  ★★☆ Tier 2 : ML Bull Cont + Tech Momentum + Price > EMA50 > EMA200 (sector not bullish)
  Note: Bull Cont only | No reversals | Fund Score >= 40.0 | 30-day cycle
  [LT] = also in Long Term recommendations

──────────────────────────────────────────────────────────────────────────────────────────────────────────
  [L] Large Cap  —  15 Tier1  |  18 Tier2  (showing top 10)  [LT overlap: 6]
──────────────────────────────────────────────────────────────────────────────────────────────────────────
  #    ★    Symbol       LT   Sector         Sec Trend    ML          Conf  Tech  SecRnk  CapRnk     Price          MCap   25d%   45d%
  ───  ───  ──────────── ──── ────────────── ──────────── ────────── ─────  ────  ──────  ──────  ────────  ────────────  ─────

In [18]:
# ── CELL 6 : PORTFOLIO INPUT & TRACKING ───────────────────────

LT_FILE    = os.path.join(PORTFOLIO_DIR, 'long_term_portfolio.csv')
SWING_FILE = os.path.join(PORTFOLIO_DIR, 'swing_portfolio.csv')

LT_COLS = [
    'Symbol', 'Entry_Price', 'Entry_Date', 'Quantity', 'Cap_Category',
    'Sector_Rank_At_Entry', 'Cap_Rank_At_Entry',
    'Sector_Rank_Change', 'Cap_Rank_Change',
    'Accumulation_Mode', 'Notes'
]

SWING_COLS = [
    'Symbol', 'Entry_Price', 'Entry_Date', 'Quantity', 'Cap_Category',
    'Sector_Rank_At_Entry', 'Cap_Rank_At_Entry',
    'Sector_Rank_Change', 'Cap_Rank_Change',
    'Cycle_Start_Date', 'Notes'
]

def load_portfolio(filepath, cols):
    """Load portfolio CSV. Returns empty DataFrame if not exists."""
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        for c in cols:
            if c not in df.columns:
                df[c] = None
        return df
    return pd.DataFrame(columns=cols)

def save_portfolio(df, filepath):
    df.to_csv(filepath, index=False)
    print(f"  ✅ Saved: {filepath}")

def validate_symbol(user_input):
    """
    Validate symbol with fuzzy matching.
    Returns confirmed symbol or None.
    """
    s = user_input.strip().upper()
    if s in valid_symbols:
        return s
    matches = get_close_matches(s, list(valid_symbols), n=3, cutoff=0.6)
    if not matches:
        print(f"  ❌ '{s}' not found. No close matches.")
        return None
    print(f"  ⚠️  '{s}' not found. Did you mean:")
    for i, m in enumerate(matches, 1):
        row = tech_df[tech_df['Symbol'] == m]
        if len(row) > 0:
            r = row.iloc[0]
            print(f"    {i}. {m:<14} {str(r.get('Sector','?')):<28} "
                  f"Rs{r.get('Current Price',0):.2f}  "
                  f"MCap {mcap_str(r.get('Market Cap Cr',0))}")
        else:
            print(f"    {i}. {m}")
    print(f"    0. None of these / skip")
    choice = input("  Enter number: ").strip()
    if choice in [str(i) for i in range(1, len(matches)+1)]:
        return matches[int(choice)-1]
    return None

def get_current_ranks(symbol):
    """Get current Sector Score and Cap Score from tech_df."""
    row = tech_df[tech_df['Symbol'] == symbol]
    if len(row) == 0:
        return None, None
    r = row.iloc[0]
    return (float(r.get('Sector Score', 0) or 0),
            float(r.get('Cap Score',    0) or 0))

def update_rank_changes(df):
    """
    Recompute Sector_Rank_Change and Cap_Rank_Change
    for all holdings vs current tech_df values.
    """
    for idx, row in df.iterrows():
        sym          = row['Symbol']
        curr_sec, curr_cap = get_current_ranks(sym)
        entry_sec    = float(row.get('Sector_Rank_At_Entry', 0) or 0)
        entry_cap    = float(row.get('Cap_Rank_At_Entry',    0) or 0)
        if curr_sec is not None:
            df.at[idx, 'Sector_Rank_Change'] = round(curr_sec - entry_sec, 1)
            df.at[idx, 'Cap_Rank_Change']    = round(curr_cap - entry_cap, 1)
    return df

def input_stock(portfolio_type, is_swing=False):
    """
    Interactively input one stock entry.
    Returns dict or None.
    """
    sym_input = input("\n  Symbol (or 'done'): ").strip()
    if sym_input.lower() == 'done':
        return 'done'

    symbol = validate_symbol(sym_input)
    if symbol is None:
        print("  Skipping.")
        return None

    info = tech_df[tech_df['Symbol'] == symbol]
    if len(info) > 0:
        r = info.iloc[0]
        print(f"  ✅ {symbol} — {r.get('Sector','?')} | "
              f"Rs{r.get('Current Price',0):.2f} | "
              f"MCap {mcap_str(r.get('Market Cap Cr',0))} | "
              f"ML: {r.get('ML_Prediction','')} | "
              f"Setup: {r.get('Best Setup','')}")

    try:
        entry_price = float(input(f"  Entry price (Rs): ").strip())
    except:
        print("  Invalid price. Skipping.")
        return None

    try:
        quantity = int(input(f"  Quantity (shares): ").strip())
    except:
        print("  Invalid quantity. Skipping.")
        return None

    entry_date = input(f"  Entry date (YYYY-MM-DD) [Enter for today]: ").strip()
    if not entry_date:
        entry_date = datetime.now().strftime('%Y-%m-%d')

    # Accumulation mode for LT only
    accum_mode = 'No'
    if not is_swing:
        accum = input(f"  Accumulation mode? (y/n) [slow avg down — exit suppressed]: ").strip().lower()
        accum_mode = 'Yes' if accum == 'y' else 'No'

    # Cycle start for swing only
    cycle_start = None
    if is_swing:
        cycle_start = input(f"  Cycle start date (YYYY-MM-DD) [Enter for today]: ").strip()
        if not cycle_start:
            cycle_start = datetime.now().strftime('%Y-%m-%d')

    notes = input(f"  Notes (optional): ").strip()

    # Get current ranks as entry baseline
    curr_sec, curr_cap = get_current_ranks(symbol)
    cap_cat = str(info.iloc[0].get('Cap Category', '')) if len(info) > 0 else ''

    record = {
        'Symbol'              : symbol,
        'Entry_Price'         : entry_price,
        'Entry_Date'          : entry_date,
        'Quantity'            : quantity,
        'Cap_Category'        : cap_cat,
        'Sector_Rank_At_Entry': curr_sec or 0,
        'Cap_Rank_At_Entry'   : curr_cap or 0,
        'Sector_Rank_Change'  : 0.0,
        'Cap_Rank_Change'     : 0.0,
        'Notes'               : notes,
    }

    if not is_swing:
        record['Accumulation_Mode'] = accum_mode
    else:
        record['Cycle_Start_Date'] = cycle_start

    invested = entry_price * quantity
    print(f"  ✅ Added {symbol} — {quantity} shares @ Rs{entry_price:.2f} "
          f"= Rs{invested:,.0f}  "
          f"SecRnk={curr_sec:.1f}  CapRnk={curr_cap:.1f}  "
          f"Accum={'Yes' if not is_swing and accum_mode=='Yes' else 'N/A' if is_swing else 'No'}")
    return record

def manage_portfolio(filepath, cols, portfolio_type, is_swing=False):
    """
    Full portfolio management — load, update, save.
    Returns final DataFrame.
    """
    df = load_portfolio(filepath, cols)

    if len(df) == 0:
        print(f"\n  No {portfolio_type} portfolio found.")
        choice = input(f"  Create new portfolio? (y/n): ").strip().lower()
        if choice != 'y':
            return df
    else:
        # Update rank changes on load
        df = update_rank_changes(df)
        print(f"\n  {portfolio_type} portfolio loaded — {len(df)} stocks")
        print(f"\n  Current holdings:")
        for i, row in df.iterrows():
            curr_sec, curr_cap = get_current_ranks(row['Symbol'])
            sec_chg = float(row.get('Sector_Rank_Change', 0) or 0)
            cap_chg = float(row.get('Cap_Rank_Change',    0) or 0)
            accum   = f" [ACCUM]" if str(row.get('Accumulation_Mode','')) == 'Yes' else ''
            print(f"    {i+1:>2}. {row['Symbol']:<14} "
                  f"Rs{float(row['Entry_Price']):.2f} × {int(row['Quantity'])}  "
                  f"SecRnk: {float(row.get('Sector_Rank_At_Entry',0)):.1f}"
                  f"({sec_chg:+.1f})  "
                  f"CapRnk: {float(row.get('Cap_Rank_At_Entry',0)):.1f}"
                  f"({cap_chg:+.1f})"
                  f"{accum}")

        print(f"\n  Options:")
        print(f"  1. Add new stocks")
        print(f"  2. Remove stocks (exited positions)")
        print(f"  3. Update accumulation mode")
        print(f"  4. No changes — proceed to review")
        sub = input("  Choice (1/2/3/4): ").strip()

        if sub == '1':
            pass  # fall through to add stocks below
        elif sub == '2':
            to_remove = input("  Enter row numbers to remove (comma separated): ").strip()
            try:
                indices = [int(x.strip())-1 for x in to_remove.split(',')]
                df      = df.drop(df.index[indices]).reset_index(drop=True)
                df      = update_rank_changes(df)
                save_portfolio(df, filepath)
            except:
                print("  Invalid input. No changes made.")
            return df
        elif sub == '3' and not is_swing:
            sym_upd = input("  Symbol to update: ").strip().upper()
            mask    = df['Symbol'] == sym_upd
            if mask.any():
                curr = df.loc[mask, 'Accumulation_Mode'].values[0]
                new  = 'No' if curr == 'Yes' else 'Yes'
                df.loc[mask, 'Accumulation_Mode'] = new
                save_portfolio(df, filepath)
                print(f"  {sym_upd} Accumulation_Mode → {new}")
            else:
                print(f"  {sym_upd} not found in portfolio.")
            return df
        elif sub == '4':
            return df

    # Add new stocks
    print(f"\n  Enter stocks to add. Type 'done' when finished.")
    new_records = []
    while True:
        record = input_stock(portfolio_type, is_swing=is_swing)
        if record == 'done':
            break
        if record is not None:
            new_records.append(record)

    if new_records:
        new_df = pd.DataFrame(new_records)
        df     = pd.concat([df, new_df], ignore_index=True)
        df     = df.drop_duplicates(subset=['Symbol'], keep='last')
        df     = update_rank_changes(df)
        save_portfolio(df, filepath)

    return df

# ── TEST LOAD ──────────────────────────────────────────────────
print("Portfolio management functions ready.")
print(f"  LT file    : {LT_FILE}")
print(f"  Swing file : {SWING_FILE}")
print(f"\n  LT columns    : {LT_COLS}")
print(f"  Swing columns : {SWING_COLS}")

# Quick test — verify rank lookup works
test_sym = 'LLOYDSME'
sec, cap = get_current_ranks(test_sym)
print(f"\n  Rank lookup test ({test_sym}): SecRnk={sec}  CapRnk={cap}")

Portfolio management functions ready.
  LT file    : E:\learn\Project 1 AI Screener\stock-ai-india\data\portfolio\long_term_portfolio.csv
  Swing file : E:\learn\Project 1 AI Screener\stock-ai-india\data\portfolio\swing_portfolio.csv

  LT columns    : ['Symbol', 'Entry_Price', 'Entry_Date', 'Quantity', 'Cap_Category', 'Sector_Rank_At_Entry', 'Cap_Rank_At_Entry', 'Sector_Rank_Change', 'Cap_Rank_Change', 'Accumulation_Mode', 'Notes']
  Swing columns : ['Symbol', 'Entry_Price', 'Entry_Date', 'Quantity', 'Cap_Category', 'Sector_Rank_At_Entry', 'Cap_Rank_At_Entry', 'Sector_Rank_Change', 'Cap_Rank_Change', 'Cycle_Start_Date', 'Notes']

  Rank lookup test (LLOYDSME): SecRnk=10.0  CapRnk=10.0


In [19]:
# ── CELL 7 : LONG TERM PORTFOLIO REVIEW ───────────────────────

def run_lt_review(lt_df):
    """
    Generate Long Term portfolio review report.
    Hold / Add / Exit recommendation per stock.
    """
    if len(lt_df) == 0:
        print("  No LT portfolio to review.")
        return []

    lt_review_lines = []
    def p(line=''): lt_review_lines.append(str(line))

    today_file = datetime.now().strftime('%Y%m%d')

    p("=" * 100)
    p("  LONG TERM PORTFOLIO — HOLDINGS REVIEW")
    p(f"  Generated : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    p(f"  Holdings  : {len(lt_df)} stocks")
    p()
    p("  Exit triggers  : Sector+Cap rank both dropped significantly OR")
    p("                   ML turned bearish + forecast negative")
    p("  Add triggers   : Rank improved + ML still strong + Tech strong")
    p("  Accum mode [A] : Price-based exit suppressed — fundamentals only")
    p("=" * 100)

    total_invested = 0
    total_curr_val = 0

    for _, holding in lt_df.iterrows():
        symbol     = str(holding['Symbol'])
        entry_px   = float(holding.get('Entry_Price', 0) or 0)
        qty        = int(holding.get('Quantity', 0) or 0)
        entry_date = str(holding.get('Entry_Date', '') or '')
        entry_sec  = float(holding.get('Sector_Rank_At_Entry', 0) or 0)
        entry_cap  = float(holding.get('Cap_Rank_At_Entry',    0) or 0)
        sec_chg    = float(holding.get('Sector_Rank_Change',   0) or 0)
        cap_chg    = float(holding.get('Cap_Rank_Change',      0) or 0)
        accum_mode = str(holding.get('Accumulation_Mode', 'No') or 'No')
        notes      = str(holding.get('Notes', '') or '')
        invested   = entry_px * qty
        total_invested += invested

        info = tech_df[tech_df['Symbol'] == symbol]
        if len(info) == 0:
            p(f"\n  {'─'*96}")
            p(f"  {symbol}  ⚠️  Not found in current universe — review manually")
            p(f"  Entry : Rs{entry_px:.2f} × {qty} = Rs{invested:,.0f}")
            p(f"  ► ACTION : 🔴 REVIEW MANUALLY")
            continue

        r          = info.iloc[0]
        curr_px    = float(r.get('Current Price',   0) or 0)
        ml_pred    = str(r.get('ML_Prediction',     ''))
        ml_conf    = float(r.get('ML_Confidence',   0) or 0)
        best_setup = str(r.get('Best Setup',        ''))
        tech_score = float(r.get('Tech Score',      0) or 0)
        tier       = str(r.get('Tier',              ''))
        rsi        = float(r.get('RSI',             0) or 0)
        adx        = float(r.get('ADX',             0) or 0)
        macd       = float(r.get('MACD Hist',       0) or 0)
        curr_sec   = float(r.get('Sector Score',    0) or 0)
        curr_cap   = float(r.get('Cap Score',       0) or 0)
        sec_trend  = str(r.get('Sector Trend',      ''))
        forecast25 = float(r.get('Forecast_25d_Pct',0) or 0)
        forecast45 = float(r.get('Forecast_45d_Pct',0) or 0)
        bull_prob  = float(r.get('Bullish_Cont_Prob',0) or 0)
        top_prob   = float(r.get('Top_Rev_Prob',    0) or 0)
        ema50      = float(r.get('EMA50',           0) or 0)
        ema200     = float(r.get('EMA200',          0) or 0)

        curr_val   = curr_px * qty
        total_curr_val += curr_val
        pl_abs     = curr_val - invested
        pl_pct     = (pl_abs / invested * 100) if invested > 0 else 0

        # Days held
        try:
            held_days = (datetime.now() -
                         datetime.strptime(entry_date, '%Y-%m-%d')).days
        except:
            held_days = 0

        # ── RECOMMENDATION LOGIC ──────────────────────────────
        exit_reasons = []
        add_reasons  = []
        warn_reasons = []

        # Rank-based exit — both ranks dropped significantly
        rank_deteriorated = sec_chg <= -1.5 and cap_chg <= -1.5
        if rank_deteriorated:
            exit_reasons.append(
                f"Both ranks dropped — SecRnk: {entry_sec:.1f}→{curr_sec:.1f} "
                f"({sec_chg:+.1f})  CapRnk: {entry_cap:.1f}→{curr_cap:.1f} "
                f"({cap_chg:+.1f})")

        # ML turned bearish with high confidence
        if ml_pred in ('Bearish Continual', 'Bearish') and ml_conf >= 50:
            exit_reasons.append(
                f"ML bearish: {ml_pred} ({ml_conf:.1f}%)")

        # Both forecasts negative
        if forecast25 < -3 and forecast45 < -3:
            exit_reasons.append(
                f"Forecast negative: 25d={forecast25:+.1f}%  45d={forecast45:+.1f}%")

        # High top reversal risk
        if top_prob >= 65 and ml_conf < 40:
            warn_reasons.append(
                f"Top reversal risk: {top_prob:.1f}% — watch closely")

        # Price below EMA200 — warning only (not exit for LT)
        if curr_px < ema200 and accum_mode != 'Yes':
            warn_reasons.append(
                f"Price below EMA200 (Rs{ema200:.2f}) — "
                f"trend weakening")

        # Add triggers
        if sec_chg >= 1.0 and cap_chg >= 1.0:
            add_reasons.append(
                f"Both ranks improved — SecRnk: {entry_sec:.1f}→{curr_sec:.1f} "
                f"({sec_chg:+.1f})  CapRnk: {entry_cap:.1f}→{curr_cap:.1f} "
                f"({cap_chg:+.1f})")
        if ml_pred == 'Bullish Continual' and ml_conf >= 65:
            add_reasons.append(
                f"Strong ML: {ml_pred} ({ml_conf:.1f}%)")
        if tech_score >= 70 and best_setup == 'Momentum':
            add_reasons.append(
                f"Strong tech: {best_setup} score={tech_score:.0f}")
        if is_sector_bullish(r) and best_setup == 'Momentum':
            add_reasons.append(
                f"Sector + stock both bullish ✅")

        # Accumulation mode — suppress price exit
        if accum_mode == 'Yes':
            # Remove price/forecast exits — keep only rank+ML exits
            exit_reasons = [e for e in exit_reasons
                            if 'Forecast' not in e]
            warn_reasons.append(
                f"Accumulation mode ON — price dips expected, "
                f"averaging down strategy")

        # Final action
        if exit_reasons:
            action = "🔴 EXIT"
        elif len(add_reasons) >= 2:
            action = "🟢 ADD"
        elif add_reasons:
            action = "🟡 HOLD  (weak add signal)"
        else:
            action = "🟡 HOLD"

        accum_tag = " [A]" if accum_mode == 'Yes' else ""

        p(f"\n  {'─'*96}")
        p(f"  {symbol}{accum_tag}  |  "
          f"{short_sector(r.get('Sector','?'))}  |  "
          f"{r.get('Cap Category','?')}  |  "
          f"MCap {mcap_str(r.get('Market Cap Cr',0))}  |  "
          f"Held: {held_days}d")
        p(f"  Entry   : Rs{entry_px:.2f} × {qty} shares on {entry_date} "
          f"= Rs{invested:,.0f}")
        p(f"  Current : Rs{curr_px:.2f}  |  "
          f"Value: Rs{curr_val:,.0f}  |  "
          f"P&L: Rs{pl_abs:+,.0f} ({pl_pct:+.1f}%)")
        p(f"  Ranks   : SecRnk {entry_sec:.1f}→{curr_sec:.1f} "
          f"({sec_chg:+.1f})  |  "
          f"CapRnk {entry_cap:.1f}→{curr_cap:.1f} ({cap_chg:+.1f})")
        p(f"  ML      : {ml_pred}  Conf:{ml_conf:.1f}%  "
          f"BullProb:{bull_prob:.1f}%  TopRevRisk:{top_prob:.1f}%")
        p(f"  Forecast: 25d={forecast25:+.1f}%  45d={forecast45:+.1f}%")
        p(f"  Tech    : Score={tech_score:.0f}  "
          f"Setup={best_setup}  "
          f"Tier={tier_abbr(tier)}  "
          f"RSI={rsi:.0f}  ADX={adx:.0f}  MACD={macd:+.2f}")
        p(f"  Sector  : {short_trend(sec_trend)}")
        if notes:
            p(f"  Notes   : {notes}")

        p(f"\n  ► ACTION : {action}")
        if exit_reasons:
            for r_ in exit_reasons:
                p(f"    ✗ Exit  : {r_}")
        if add_reasons:
            for r_ in add_reasons:
                p(f"    ✓ Add   : {r_}")
        if warn_reasons:
            for r_ in warn_reasons:
                p(f"    ⚠ Warn  : {r_}")

    # Portfolio summary
    portfolio_pl     = total_curr_val - total_invested
    portfolio_pl_pct = (portfolio_pl / total_invested * 100) \
                        if total_invested > 0 else 0

    p(f"\n{'─'*100}")
    p(f"  PORTFOLIO SUMMARY")
    p(f"  Total Invested : Rs{total_invested:,.0f}")
    p(f"  Current Value  : Rs{total_curr_val:,.0f}")
    p(f"  Total P&L      : Rs{portfolio_pl:+,.0f} ({portfolio_pl_pct:+.1f}%)")
    p(f"{'─'*100}")
    p(f"  [A] = Accumulation mode — price dips tolerated, averaging strategy")
    p(f"  Exit based on rank deterioration + ML signal, not price alone")
    p(f"{'─'*100}")

    return lt_review_lines

# ── TEST WITH SAMPLE DATA ──────────────────────────────────────
# Create a small test portfolio with 2 stocks to verify logic
test_lt = pd.DataFrame([
    {
        'Symbol'              : 'LLOYDSME',
        'Entry_Price'         : 1400.00,
        'Entry_Date'          : '2026-03-01',
        'Quantity'            : 10,
        'Cap_Category'        : 'Large Cap',
        'Sector_Rank_At_Entry': 9.0,
        'Cap_Rank_At_Entry'   : 9.0,
        'Sector_Rank_Change'  : 0.0,
        'Cap_Rank_Change'     : 0.0,
        'Accumulation_Mode'   : 'No',
        'Notes'               : 'Metals play',
    },
    {
        'Symbol'              : 'NATIONALUM',
        'Entry_Price'         : 450.00,
        'Entry_Date'          : '2026-02-15',
        'Quantity'            : 50,
        'Cap_Category'        : 'Large Cap',
        'Sector_Rank_At_Entry': 9.5,
        'Cap_Rank_At_Entry'   : 9.5,
        'Sector_Rank_Change'  : 0.0,
        'Cap_Rank_Change'     : 0.0,
        'Accumulation_Mode'   : 'Yes',
        'Notes'               : 'Slow accumulation',
    },
])

# Update rank changes from current tech_df
test_lt = update_rank_changes(test_lt)
print("Test portfolio after rank update:")
for _, r in test_lt.iterrows():
    print(f"  {r['Symbol']:<14} "
          f"SecRnk: {r['Sector_Rank_At_Entry']:.1f}→"
          f"{r['Sector_Rank_At_Entry']+r['Sector_Rank_Change']:.1f} "
          f"({r['Sector_Rank_Change']:+.1f})  "
          f"CapRnk: {r['Cap_Rank_At_Entry']:.1f}→"
          f"{r['Cap_Rank_At_Entry']+r['Cap_Rank_Change']:.1f} "
          f"({r['Cap_Rank_Change']:+.1f})  "
          f"Accum={r['Accumulation_Mode']}")

print()
lines = run_lt_review(test_lt)
for line in lines:
    print(line)

Test portfolio after rank update:
  LLOYDSME       SecRnk: 9.0→10.0 (+1.0)  CapRnk: 9.0→10.0 (+1.0)  Accum=No
  NATIONALUM     SecRnk: 9.5→9.2 (-0.3)  CapRnk: 9.5→9.2 (-0.3)  Accum=Yes

  LONG TERM PORTFOLIO — HOLDINGS REVIEW
  Generated : 2026-04-15 10:47
  Holdings  : 2 stocks

  Exit triggers  : Sector+Cap rank both dropped significantly OR
                   ML turned bearish + forecast negative
  Add triggers   : Rank improved + ML still strong + Tech strong
  Accum mode [A] : Price-based exit suppressed — fundamentals only

  ────────────────────────────────────────────────────────────────────────────────────────────────
  LLOYDSME  |  Metals  |  Large Cap  |  MCap Rs70,094Cr  |  Held: 45d
  Entry   : Rs1400.00 × 10 shares on 2026-03-01 = Rs14,000
  Current : Rs1510.00  |  Value: Rs15,100  |  P&L: Rs+1,100 (+7.9%)
  Ranks   : SecRnk 9.0→10.0 (+1.0)  |  CapRnk 9.0→10.0 (+1.0)
  ML      : Bullish Continual  Conf:76.3%  BullProb:76.3%  TopRevRisk:78.1%
  Forecast: 25d=+2.2%  45d=+3.

In [20]:
# ── CELL 8 : SWING PORTFOLIO REVIEW ───────────────────────────

SWING_CAPITAL    = 150000   # Rs 1.5 Lakhs
DRAWDOWN_LIMIT   = 0.08     # 8% portfolio drawdown
MAX_HOLD_DAYS    = 30       # 30-day cycle
MIN_CARRY_CONF   = 65.0     # Min ML conf to carry forward
MIN_CARRY_F25    = 0.0      # Min forecast to carry forward

def compute_swing_allocation(swing_df):
    """
    Weighted allocation: 50% SecRnk + 50% CapRnk
    Normalized to SWING_CAPITAL.
    """
    weights  = []
    symbols  = []
    for _, row in swing_df.iterrows():
        sym  = str(row['Symbol'])
        info = tech_df[tech_df['Symbol'] == sym]
        if len(info) > 0:
            r = info.iloc[0]
            w = (float(r.get('Sector Score', 5) or 5) * 0.5 +
                 float(r.get('Cap Score',    5) or 5) * 0.5)
        else:
            w = 5.0
        weights.append(w)
        symbols.append(sym)
    total_w = sum(weights) if sum(weights) > 0 else 1
    allocs  = {sym: round(SWING_CAPITAL * w / total_w)
               for sym, w in zip(symbols, weights)}
    alloc_p = {sym: round(w / total_w * 100, 1)
               for sym, w in zip(symbols, weights)}
    return allocs, alloc_p

def run_swing_review(swing_df):
    """
    Generate Swing portfolio review report.
    Hold / Add / Exit / Carry Forward per stock.
    Portfolio-level drawdown check.
    """
    if len(swing_df) == 0:
        print("  No swing portfolio to review.")
        return []

    swing_review_lines = []
    def p(line=''): swing_review_lines.append(str(line))

    today_file = datetime.now().strftime('%Y%m%d')

    p("=" * 100)
    p("  SWING TRADING PORTFOLIO — HOLDINGS REVIEW")
    p(f"  Generated  : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    p(f"  Holdings   : {len(swing_df)} stocks")
    p(f"  Capital    : Rs{SWING_CAPITAL:,.0f}  |  "
      f"Drawdown limit: {DRAWDOWN_LIMIT*100:.0f}%  |  "
      f"Cycle: 30 days")
    p()
    p("  Exit triggers   : ML no longer Bull Cont OR forecast negative OR")
    p("                    portfolio drawdown >= 8% (exit weakest first)")
    p("  Carry Forward   : 30d done but ML conf >= 65% + forecast positive")
    p("  Add triggers    : ML conf rising + Tech strong + Sector bullish")
    p("=" * 100)

    # ── SECTION 1: PORTFOLIO P&L + DRAWDOWN ──────────────────
    total_invested = 0
    total_curr_val = 0
    stock_data     = []

    for _, holding in swing_df.iterrows():
        symbol   = str(holding['Symbol'])
        entry_px = float(holding.get('Entry_Price', 0) or 0)
        qty      = int(holding.get('Quantity', 0) or 0)
        invested = entry_px * qty
        total_invested += invested

        info = tech_df[tech_df['Symbol'] == symbol]
        if len(info) > 0:
            curr_px  = float(info.iloc[0].get('Current Price', 0) or 0)
            ml_conf  = float(info.iloc[0].get('ML_Confidence', 0) or 0)
            f25      = float(info.iloc[0].get('Forecast_25d_Pct', 0) or 0)
        else:
            curr_px  = entry_px
            ml_conf  = 0.0
            f25      = 0.0

        curr_val = curr_px * qty
        total_curr_val += curr_val
        pl_pct   = ((curr_val - invested) / invested * 100) \
                    if invested > 0 else 0

        stock_data.append({
            'symbol'  : symbol,
            'invested': invested,
            'curr_val': curr_val,
            'pl_pct'  : pl_pct,
            'ml_conf' : ml_conf,
            'f25'     : f25,
            'holding' : holding,
        })

    portfolio_pl_pct = ((total_curr_val - total_invested) /
                         total_invested * 100) \
                        if total_invested > 0 else 0
    drawdown_hit     = portfolio_pl_pct <= -(DRAWDOWN_LIMIT * 100)

    # Sort by ML confidence ascending — weakest first for drawdown exit
    stock_data_sorted = sorted(stock_data, key=lambda x: x['ml_conf'])

    p(f"\n{'─'*100}")
    p(f"  SECTION 1 — PORTFOLIO P&L & DRAWDOWN")
    p(f"{'─'*100}")
    p(f"  Total Invested : Rs{total_invested:,.0f}")
    p(f"  Current Value  : Rs{total_curr_val:,.0f}")
    p(f"  Portfolio P&L  : Rs{total_curr_val - total_invested:+,.0f} "
      f"({portfolio_pl_pct:+.1f}%)")
    p(f"  Drawdown limit : {DRAWDOWN_LIMIT*100:.0f}%")

    if drawdown_hit:
        p(f"\n  ⚠️  DRAWDOWN ALERT — Portfolio down "
          f"{abs(portfolio_pl_pct):.1f}%")
        p(f"     Action: Exit weakest stock first "
          f"(lowest ML confidence)")
        p(f"     Weakest: {stock_data_sorted[0]['symbol']} "
          f"(ML conf: {stock_data_sorted[0]['ml_conf']:.1f}%)")
    else:
        remaining_room = DRAWDOWN_LIMIT*100 - abs(min(portfolio_pl_pct, 0))
        p(f"  Status         : ✅ Within limit  "
          f"(room remaining: {remaining_room:.1f}%)")

    # Allocation guide
    allocs, alloc_p = compute_swing_allocation(swing_df)
    p(f"\n{'─'*100}")
    p(f"  SECTION 2 — ALLOCATION GUIDE")
    p(f"  Weight = 50% SecRnk + 50% CapRnk | Capital: Rs{SWING_CAPITAL:,.0f}")
    p(f"{'─'*100}")
    p(f"  {'Symbol':<14} {'Cap':<16} {'SecRnk':>6}  {'CapRnk':>6}  "
      f"{'Alloc%':>6}  {'Rs Amount':>10}  {'Invested':>10}  {'P&L%':>6}")
    p(f"  {'─'*14} {'─'*16} {'─'*6}  {'─'*6}  "
      f"{'─'*6}  {'─'*10}  {'─'*10}  {'─'*6}")

    for sd in stock_data:
        sym  = sd['symbol']
        info = tech_df[tech_df['Symbol'] == sym]
        sec_r= float(info.iloc[0].get('Sector Score', 0) or 0) \
               if len(info) > 0 else 0
        cap_r= float(info.iloc[0].get('Cap Score', 0) or 0) \
               if len(info) > 0 else 0
        cap_c= str(info.iloc[0].get('Cap Category', '') or '') \
               if len(info) > 0 else ''
        p(f"  {sym:<14} {cap_c:<16} {sec_r:>6.1f}  {cap_r:>6.1f}  "
          f"{alloc_p.get(sym,0):>6.1f}%  "
          f"Rs{allocs.get(sym,0):>8,.0f}  "
          f"Rs{sd['invested']:>8,.0f}  "
          f"{sd['pl_pct']:>+6.1f}%")

    # ── SECTION 3: STOCK-BY-STOCK REVIEW ─────────────────────
    p(f"\n{'─'*100}")
    p(f"  SECTION 3 — STOCK-BY-STOCK REVIEW")
    p(f"  (Sorted by ML confidence — weakest first for drawdown decisions)")
    p(f"{'─'*100}")

    weakest_flagged = False

    for sd in stock_data_sorted:
        symbol     = sd['symbol']
        holding    = sd['holding']
        entry_px   = float(holding.get('Entry_Price',  0) or 0)
        qty        = int(holding.get('Quantity',        0) or 0)
        entry_date = str(holding.get('Entry_Date',     '') or '')
        entry_sec  = float(holding.get('Sector_Rank_At_Entry', 0) or 0)
        entry_cap  = float(holding.get('Cap_Rank_At_Entry',    0) or 0)
        sec_chg    = float(holding.get('Sector_Rank_Change',   0) or 0)
        cap_chg    = float(holding.get('Cap_Rank_Change',      0) or 0)
        cycle_start= str(holding.get('Cycle_Start_Date', entry_date) or entry_date)
        notes      = str(holding.get('Notes', '') or '')
        invested   = sd['invested']
        curr_val   = sd['curr_val']
        pl_pct     = sd['pl_pct']

        # Days in cycle
        try:
            cycle_days = (datetime.now() -
                          datetime.strptime(cycle_start, '%Y-%m-%d')).days
        except:
            cycle_days = 0

        info = tech_df[tech_df['Symbol'] == symbol]
        if len(info) == 0:
            p(f"\n  {'─'*96}")
            p(f"  {symbol}  ⚠️  Not found in current universe")
            p(f"  ► ACTION : 🔴 EXIT — symbol removed from universe")
            continue

        r          = info.iloc[0]
        curr_px    = float(r.get('Current Price',    0) or 0)
        ml_pred    = str(r.get('ML_Prediction',      ''))
        ml_conf    = float(r.get('ML_Confidence',    0) or 0)
        best_setup = str(r.get('Best Setup',         ''))
        tech_score = float(r.get('Tech Score',       0) or 0)
        tier       = str(r.get('Tier',               ''))
        rsi        = float(r.get('RSI',              0) or 0)
        adx        = float(r.get('ADX',              0) or 0)
        macd       = float(r.get('MACD Hist',        0) or 0)
        curr_sec   = float(r.get('Sector Score',     0) or 0)
        curr_cap   = float(r.get('Cap Score',        0) or 0)
        sec_trend  = str(r.get('Sector Trend',       ''))
        forecast25 = float(r.get('Forecast_25d_Pct', 0) or 0)
        forecast45 = float(r.get('Forecast_45d_Pct', 0) or 0)
        bull_prob  = float(r.get('Bullish_Cont_Prob',0) or 0)
        top_prob   = float(r.get('Top_Rev_Prob',     0) or 0)
        ema_ok     = passes_ema_filter(r)

        # ── SWING EXIT/HOLD/ADD LOGIC ─────────────────────────
        exit_reasons    = []
        add_reasons     = []
        warn_reasons    = []
        carry_forward   = False

        # Hard exit: ML no longer Bullish Continual
        if ml_pred != 'Bullish Continual':
            exit_reasons.append(
                f"ML changed: {ml_pred} — no longer Bullish Continual")

        # Hard exit: forecast turned negative
        if forecast25 < 0 and forecast45 < 0:
            exit_reasons.append(
                f"Forecast negative: 25d={forecast25:+.1f}%  "
                f"45d={forecast45:+.1f}%")

        # Hard exit: EMA structure broken
        if not ema_ok and ml_pred == 'Bullish Continual':
            exit_reasons.append(
                f"EMA structure broken — Price no longer > EMA50 > EMA200")

        # Drawdown exit — weakest stock
        if drawdown_hit and not weakest_flagged:
            exit_reasons.append(
                f"Portfolio drawdown {abs(portfolio_pl_pct):.1f}% — "
                f"exit weakest stock (lowest ML conf)")
            weakest_flagged = True

        # 30-day cycle complete
        if cycle_days >= MAX_HOLD_DAYS:
            if (ml_conf >= MIN_CARRY_CONF and
                forecast25 >= MIN_CARRY_F25 and
                ml_pred == 'Bullish Continual' and
                ema_ok):
                carry_forward = True
            else:
                exit_reasons.append(
                    f"30-day cycle complete ({cycle_days}d) — "
                    f"setup not strong enough to carry forward  "
                    f"(need conf>={MIN_CARRY_CONF}% + forecast>0)")

        # Rank deterioration warning
        if sec_chg <= -1.0 or cap_chg <= -1.0:
            warn_reasons.append(
                f"Rank declining — SecRnk: {entry_sec:.1f}→{curr_sec:.1f} "
                f"({sec_chg:+.1f})  "
                f"CapRnk: {entry_cap:.1f}→{curr_cap:.1f} ({cap_chg:+.1f})")

        # Top reversal risk warning
        if top_prob >= 65:
            warn_reasons.append(
                f"Top reversal risk: {top_prob:.1f}% — consider tightening stop")

        # Add triggers
        if (ml_conf >= 75 and forecast25 > 3 and
                ml_pred == 'Bullish Continual'):
            add_reasons.append(
                f"Strong ML: conf={ml_conf:.1f}%  forecast={forecast25:+.1f}%")
        if is_sector_bullish(r) and best_setup == 'Momentum':
            add_reasons.append(
                f"Sector + stock both bullish ✅")
        if tech_score >= 75 and best_setup == 'Momentum':
            add_reasons.append(
                f"Strong tech: score={tech_score:.0f}")
        if sec_chg >= 1.0 and cap_chg >= 1.0:
            add_reasons.append(
                f"Both ranks improved ({sec_chg:+.1f} / {cap_chg:+.1f})")

        # Final action
        if carry_forward:
            action = "🔵 CARRY FORWARD"
        elif exit_reasons:
            action = "🔴 EXIT"
        elif len(add_reasons) >= 2:
            action = "🟢 ADD"
        elif add_reasons:
            action = "🟡 HOLD  (mild add signal)"
        else:
            action = "🟡 HOLD"

        p(f"\n  {'─'*96}")
        p(f"  {symbol}  |  "
          f"{short_sector(r.get('Sector','?'))}  |  "
          f"{r.get('Cap Category','?')}  |  "
          f"MCap {mcap_str(r.get('Market Cap Cr',0))}  |  "
          f"Cycle day: {cycle_days}/{MAX_HOLD_DAYS}")
        p(f"  Entry   : Rs{entry_px:.2f} × {qty} shares on {entry_date} "
          f"= Rs{invested:,.0f}")
        p(f"  Current : Rs{curr_px:.2f}  |  "
          f"Value: Rs{curr_val:,.0f}  |  "
          f"P&L: Rs{curr_val-invested:+,.0f} ({pl_pct:+.1f}%)")
        p(f"  Ranks   : SecRnk {entry_sec:.1f}→{curr_sec:.1f} "
          f"({sec_chg:+.1f})  |  "
          f"CapRnk {entry_cap:.1f}→{curr_cap:.1f} ({cap_chg:+.1f})")
        p(f"  ML      : {ml_pred}  Conf:{ml_conf:.1f}%  "
          f"BullProb:{bull_prob:.1f}%  TopRevRisk:{top_prob:.1f}%")
        p(f"  Forecast: 25d={forecast25:+.1f}%  45d={forecast45:+.1f}%")
        p(f"  Tech    : Score={tech_score:.0f}  "
          f"Setup={best_setup}  "
          f"Tier={tier_abbr(tier)}  "
          f"RSI={rsi:.0f}  ADX={adx:.0f}  MACD={macd:+.2f}")
        p(f"  Sector  : {short_trend(sec_trend)}  |  "
          f"EMA filter: {'✅ OK' if ema_ok else '❌ Broken'}")
        if notes:
            p(f"  Notes   : {notes}")

        p(f"\n  ► ACTION : {action}")
        if carry_forward:
            p(f"    🔵 Setup still strong — carry to next 30-day cycle")
            p(f"       ML conf={ml_conf:.1f}%  Forecast={forecast25:+.1f}%  "
              f"EMA OK={ema_ok}")
        elif exit_reasons:
            for r_ in exit_reasons:
                p(f"    ✗ Exit  : {r_}")
        if add_reasons and not carry_forward:
            for r_ in add_reasons:
                p(f"    ✓ Add   : {r_}")
        if warn_reasons:
            for r_ in warn_reasons:
                p(f"    ⚠ Warn  : {r_}")

    p(f"\n{'─'*100}")
    p(f"  PORTFOLIO SUMMARY")
    p(f"  Total Invested : Rs{total_invested:,.0f}")
    p(f"  Current Value  : Rs{total_curr_val:,.0f}")
    p(f"  Total P&L      : Rs{total_curr_val - total_invested:+,.0f} "
      f"({portfolio_pl_pct:+.1f}%)")
    p(f"  Drawdown status: "
      f"{'⚠️  ALERT — exit weakest stock' if drawdown_hit else '✅ Within limit'}")
    p(f"{'─'*100}")
    p(f"  🔵 CARRY FORWARD = continue to next 30-day cycle")
    p(f"  🔴 EXIT = close position")
    p(f"  🟢 ADD  = increase position")
    p(f"  🟡 HOLD = maintain position")
    p(f"{'─'*100}")

    return swing_review_lines

# ── TEST WITH SAMPLE DATA ──────────────────────────────────────
test_swing = pd.DataFrame([
    {
        'Symbol'              : 'LLOYDSME',
        'Entry_Price'         : 1480.00,
        'Entry_Date'          : '2026-03-20',
        'Quantity'            : 5,
        'Cap_Category'        : 'Large Cap',
        'Sector_Rank_At_Entry': 9.5,
        'Cap_Rank_At_Entry'   : 9.5,
        'Sector_Rank_Change'  : 0.0,
        'Cap_Rank_Change'     : 0.0,
        'Cycle_Start_Date'    : '2026-03-20',
        'Notes'               : 'Metals momentum',
    },
    {
        'Symbol'              : 'NATCOPHARM',
        'Entry_Price'         : 1050.00,
        'Entry_Date'          : '2026-03-01',
        'Quantity'            : 10,
        'Cap_Category'        : 'Mini Large Cap',
        'Sector_Rank_At_Entry': 10.0,
        'Cap_Rank_At_Entry'   : 10.0,
        'Sector_Rank_Change'  : 0.0,
        'Cap_Rank_Change'     : 0.0,
        'Cycle_Start_Date'    : '2026-03-01',
        'Notes'               : 'Healthcare momentum',
    },
])

test_swing = update_rank_changes(test_swing)
print("Test swing portfolio:")
for _, r in test_swing.iterrows():
    print(f"  {r['Symbol']:<14} "
          f"SecRnk: {r['Sector_Rank_At_Entry']:.1f} "
          f"({r['Sector_Rank_Change']:+.1f})  "
          f"CapRnk: {r['Cap_Rank_At_Entry']:.1f} "
          f"({r['Cap_Rank_Change']:+.1f})")

print()
lines = run_swing_review(test_swing)
for line in lines:
    print(line)

Test swing portfolio:
  LLOYDSME       SecRnk: 9.5 (+0.5)  CapRnk: 9.5 (+0.5)
  NATCOPHARM     SecRnk: 10.0 (-0.3)  CapRnk: 10.0 (-0.1)

  SWING TRADING PORTFOLIO — HOLDINGS REVIEW
  Generated  : 2026-04-15 10:50
  Holdings   : 2 stocks
  Capital    : Rs150,000  |  Drawdown limit: 8%  |  Cycle: 30 days

  Exit triggers   : ML no longer Bull Cont OR forecast negative OR
                    portfolio drawdown >= 8% (exit weakest first)
  Carry Forward   : 30d done but ML conf >= 65% + forecast positive
  Add triggers    : ML conf rising + Tech strong + Sector bullish

────────────────────────────────────────────────────────────────────────────────────────────────────
  SECTION 1 — PORTFOLIO P&L & DRAWDOWN
────────────────────────────────────────────────────────────────────────────────────────────────────
  Total Invested : Rs17,900
  Current Value  : Rs18,590
  Portfolio P&L  : Rs+690 (+3.9%)
  Drawdown limit : 8%
  Status         : ✅ Within limit  (room remaining: 8.0%)

───────────────

In [21]:
# ── CELL 9 : MAIN MENU + SAVE REPORTS ─────────────────────────

today_file = datetime.now().strftime('%Y%m%d')

def save_report(lines, filepath):
    with open(filepath, 'w', encoding='utf-8') as f:
        f.write('\n'.join(lines))
    print(f"  ✅ Saved: {filepath}")

def run_long_term_full():
    """Full LT flow: recommendations → portfolio input → review → save."""
    print("\n" + "=" * 60)
    print("  LONG TERM PORTFOLIO")
    print("=" * 60)

    all_lines = []

    # Step 1: Show recommendations
    print("\n  Generating Long Term recommendations...")
    all_lines += lt_lines
    all_lines += ['', '']

    # Step 2: Portfolio management
    print("\n  Would you like to update your LT portfolio?")
    choice = input("  (y/n): ").strip().lower()
    if choice == 'y':
        lt_df = manage_portfolio(LT_FILE, LT_COLS,
                                 'Long Term', is_swing=False)
    else:
        lt_df = load_portfolio(LT_FILE, LT_COLS)
        if len(lt_df) > 0:
            lt_df = update_rank_changes(lt_df)

    # Step 3: Review if portfolio exists
    if len(lt_df) > 0:
        print(f"\n  Running LT portfolio review ({len(lt_df)} stocks)...")
        review_lines = run_lt_review(lt_df)
        all_lines   += review_lines
        for line in review_lines:
            print(line)
    else:
        print("  No LT portfolio to review.")

    # Step 4: Save report
    report_path = os.path.join(
        REPORTS_DIR, f'lt_portfolio_{today_file}.txt')
    save_report(all_lines, report_path)
    return all_lines

def run_swing_full():
    """Full Swing flow: recommendations → portfolio input → review → save."""
    print("\n" + "=" * 60)
    print("  SWING TRADING PORTFOLIO")
    print("=" * 60)

    all_lines = []

    # Step 1: Show recommendations
    print("\n  Generating Swing recommendations...")
    all_lines += swing_lines
    all_lines += ['', '']

    # Step 2: Portfolio management
    print("\n  Would you like to update your Swing portfolio?")
    choice = input("  (y/n): ").strip().lower()
    if choice == 'y':
        swing_df = manage_portfolio(SWING_FILE, SWING_COLS,
                                    'Swing', is_swing=True)
    else:
        swing_df = load_portfolio(SWING_FILE, SWING_COLS)
        if len(swing_df) > 0:
            swing_df = update_rank_changes(swing_df)

    # Step 3: Review if portfolio exists
    if len(swing_df) > 0:
        print(f"\n  Running Swing portfolio review ({len(swing_df)} stocks)...")
        review_lines = run_swing_review(swing_df)
        all_lines   += review_lines
        for line in review_lines:
            print(line)
    else:
        print("  No Swing portfolio to review.")

    # Step 4: Save report
    report_path = os.path.join(
        REPORTS_DIR, f'swing_portfolio_{today_file}.txt')
    save_report(all_lines, report_path)
    return all_lines

# ── MAIN MENU ─────────────────────────────────────────────────
print("=" * 60)
print("  AI Stock Screener — Portfolio Manager")
print(f"  Date : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("=" * 60)
print()
print("  1. Long Term Portfolio")
print("  2. Swing Trading Portfolio")
print("  3. Both")
print()
choice = input("  Enter choice (1/2/3): ").strip()

if choice == '1':
    run_long_term_full()
elif choice == '2':
    run_swing_full()
elif choice == '3':
    run_long_term_full()
    run_swing_full()
else:
    print("  Invalid choice.")

print()
print("=" * 60)
print("  Portfolio Manager complete!")
print(f"  Finished : {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print(f"  Reports  : {REPORTS_DIR}")
print("=" * 60)

  AI Stock Screener — Portfolio Manager
  Date : 2026-04-15 10:51

  1. Long Term Portfolio
  2. Swing Trading Portfolio
  3. Both



  Enter choice (1/2/3):  1



  LONG TERM PORTFOLIO

  Generating Long Term recommendations...

  Would you like to update your LT portfolio?


  (y/n):  y



  No Long Term portfolio found.


  Create new portfolio? (y/n):  n


  No LT portfolio to review.
  ✅ Saved: E:\learn\Project 1 AI Screener\stock-ai-india\data\reports\portfolio\lt_portfolio_20260415.txt

  Portfolio Manager complete!
  Finished : 2026-04-15 10:52
  Reports  : E:\learn\Project 1 AI Screener\stock-ai-india\data\reports\portfolio
